RANDOM FOREST REGRESSOR MODEL

Michael Owens 

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [2]:
# Bringing in the Data

# read data
df = pd.read_csv('LassoFeatures2.csv')
df = df.drop('Unnamed: 0',axis=1)
print(df.shape)
df.head(3)

(1788, 79)


,disc_year,release_month^2 sy_dist,release_month^2 st_mass,release_month sy_pnum sy_w4mag,release_month ttv_flag pl_tsystemref_BJD-UTC,release_month elat pl_orbper,release_month glat pl_tsystemref_JD,release_month st_nrvc sy_dist,sy_snum ttv_flag sy_w2mag,sy_snum ttv_flag pl_tsystemref_BJD-UTC,...,sy_vmag st_mass soltype_Published Confirmed,sy_pmdec st_rad^2,sy_pmra st_teff soltype_Kepler Project Candidate (q1_q17_dr25_koi),pl_orbper st_teff pl_tsystemref_BJD-TDB,pl_orbper sy_plx st_mass,st_rad soltype_Published Confirmed pl_tsystemref_BJD,sy_dist sy_w4mag sy_plx,sy_dist soltype_Published Confirmed pl_tsystemref_BJD,st_mass^2 pl_tsystemref_BJD-TDB,log radius
0,1.965840,-1.183263,-1.178747,-0.855449,-0.033464,-0.302074,-0.804590,-0.063138,-0.306723,-0.033464,...,-0.448406,-0.766246,0.047605,-0.077836,-0.215046,1.612871,-1.483149,0.620083,-0.245837,0.194514
1,1.099028,1.661502,2.533746,0.256445,-0.033464,-0.296604,-0.804590,-0.063138,-0.306723,-0.033464,...,-0.239621,0.189222,0.047605,-0.077836,-0.200082,0.882010,-0.197230,1.477871,-0.245837,0.017033
2,2.399245,-1.042761,-0.926440,-0.357918,-0.033464,-0.299938,-3.759662,-0.063138,-0.306723,-0.033464,...,-0.339190,-0.015178,0.047605,-0.077836,-0.055429,-0.424579,-0.267455,-0.354008,-0.245837,0.320977


Grid Search to Determine the Number of Estimators and Tree Depth

In [3]:
#Split data & define features/targets
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)
X_train = df_train.drop('log radius',axis=1)
X_test  = df_test.drop('log radius',axis=1)
y_train   = df_train['log radius']
y_test    = df_test['log radius']
X_train.describe()

,disc_year,release_month^2 sy_dist,release_month^2 st_mass,release_month sy_pnum sy_w4mag,release_month ttv_flag pl_tsystemref_BJD-UTC,release_month elat pl_orbper,release_month glat pl_tsystemref_JD,release_month st_nrvc sy_dist,sy_snum ttv_flag sy_w2mag,sy_snum ttv_flag pl_tsystemref_BJD-UTC,...,sy_vmag sy_bmag st_mass,sy_vmag st_mass soltype_Published Confirmed,sy_pmdec st_rad^2,sy_pmra st_teff soltype_Kepler Project Candidate (q1_q17_dr25_koi),pl_orbper st_teff pl_tsystemref_BJD-TDB,pl_orbper sy_plx st_mass,st_rad soltype_Published Confirmed pl_tsystemref_BJD,sy_dist sy_w4mag sy_plx,sy_dist soltype_Published Confirmed pl_tsystemref_BJD,st_mass^2 pl_tsystemref_BJD-TDB
count,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,...,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000,1430.000000
mean,0.003693,0.006237,0.006679,0.018389,0.008378,0.005742,-0.015823,0.000069,0.002957,0.008378,...,0.010418,-0.012670,0.002831,0.000278,0.011482,-0.004823,-0.017903,-0.001592,-0.012071,0.015335
std,1.029338,0.982458,1.010912,1.026822,1.118425,1.068424,1.058682,0.947531,0.997477,1.118425,...,0.990904,1.021083,0.967132,1.041962,1.104288,0.941554,0.981555,1.001103,0.980785,1.027095
min,-3.235030,-1.192202,-1.200693,-0.977161,-0.033464,-1.706747,-13.115605,-0.063138,-0.306723,-0.033464,...,-3.455538,-3.524744,-9.972642,-18.634236,-0.077836,-0.227008,-0.424579,-9.272644,-0.354008,-0.245837
25%,-0.201189,-0.610118,-0.409001,-0.459860,-0.033464,-0.287225,-0.804590,-0.063138,-0.306723,-0.033464,...,-0.652071,-0.401322,-0.229008,0.047605,-0.077836,-0.205233,-0.424579,-0.551013,-0.354008,-0.245837
50%,-0.201189,-0.136631,-0.269131,-0.439654,-0.033464,-0.229690,0.204050,-0.063138,-0.306723,-0.033464,...,0.112120,0.143295,0.100641,0.047605,-0.077836,-0.169199,-0.424579,0.123811,-0.354008,-0.245837
75%,-0.201189,0.321797,-0.075941,0.177108,-0.033464,-0.079752,0.620971,-0.063138,-0.306723,-0.033464,...,0.695008,0.608515,0.372748,0.047605,-0.077836,-0.072087,-0.424579,0.723587,-0.354008,-0.245837
max,3.699463,8.332578,6.437534,11.460164,29.883106,24.937303,4.539585,24.863439,6.993754,29.883106,...,5.694634,3.839665,8.041209,9.269401,23.530762,22.004695,12.673312,2.110887,9.319793,10.027180


In [5]:
grid = {'max_depth' : np.arange(1,40,5),'n_estimators':np.arange(1,5000,250)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1,verbose=3)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Fitting 5 folds for each of 160 candidates, totalling 800 fits
Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(16), 'n_estimators': np.int64(501)}
    Optimal Valid R2 = 0.39820096450170206


Implementing a Heap Map to Assist Grid Search

In [6]:
Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

In [7]:
fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Refined Grid Search

In [8]:
grid = {'max_depth' : np.arange(12,20,1),'n_estimators':np.arange(251,750,50)}
rfr2 = RandomForestRegressor(max_features = 1/3)
rfr2CV = GridSearchCV(rfr2,param_grid= grid,n_jobs=-1)
rfr2CV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfr2CV.best_params_ )
print('    Optimal Valid R2 =', rfr2CV.best_score_ )

Scores_mean = rfr2CV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Number of Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(17), 'n_estimators': np.int64(501)}
    Optimal Valid R2 = 0.4001651489923728


Evaluating the Model

In [ ]:
#results = pd.DataFrame()
#results['trees'] = grid['n_estimators']
#results['train R2'] = rfrCV.cv_results_['mean_train_score']
#results['valid R2']  = rfrCV.cv_results_['mean_test_score']
#results[['train R2','valid R2']].plot.line()

In [9]:
# test R2
print(f" train R2 {rfr2CV.score(X_train,y_train):.3f}")
print(f" test R2 {rfr2CV.score(X_test,y_test):.3f}")

 train R2 0.891
 test R2 0.510


MSE

In [10]:
from sklearn.metrics import mean_squared_error
y_pred_test = rfr2CV.predict(X_test)
y_pred_train = rfr2CV.predict(X_train)

print(mean_squared_error(y_test,y_pred_test))
print(mean_squared_error(y_train,y_pred_train))


0.03507427635157881
0.008397260359411693


In [11]:


rf = RandomForestRegressor(max_depth = 17,n_estimators=501,max_features = 1/3,oob_score=True)
rf.fit(X_train,y_train)

print(f'out-of-bag R2 = {rf.oob_score_:.3f}')
print()
print(f'training R2 {rf.score(X_train,y_train)}')
print(f'testing R2: {rf.score(X_test,y_test)}')

from sklearn.metrics import mean_squared_error
y_pred_test = rf.predict(X_test)
y_pred_train = rf.predict(X_train)

print(f'MSE Train: {mean_squared_error(y_test,y_pred_test)}')
print(f'MSE TEST: {mean_squared_error(y_train,y_pred_train)}')


out-of-bag R2 = 0.422

training R2 0.8943456504773744
testing R2: 0.5094147142240801
MSE Train: 0.035135113350475325
MSE TEST: 0.008161393276337577


In [12]:
rel_unc = abs((10**y_pred_test)-(10**y_test))/(10**y_test)
print(max(rel_unc))
print(min(rel_unc))
rel_unc.mean()

1.7238705414954802
0.00384456898281228


np.float64(0.32638249101674593)

ValueError: continuous is not supported